# Statbotics + TBA Match Exports

Give the notebook an event key (for example 2026orsal) and it will:

1. Pull match predictions and EPA/CoEPA stats from Statbotics.
2. Download each match's scoring breakdowns (hubs, fuel counts, etc.) from The Blue Alliance.
3. Produce sb.csv (Statbotics metrics) and tba.csv (TBA breakdown counts).

Set the TBA_AUTH_KEY environment variable before running the exporter cell.


In [1]:
import csv
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

import requests
from statbotics import Statbotics

TBA_BASE_URL = "https://www.thebluealliance.com/api/v3"
sb_client = Statbotics()
TEAM_MATCH_BATCH_SIZE = 1000
TEAM_METRIC_KEYS = [
    "total_points",
    "total_fuel",
    "total_tower",
    "auto_tower",
    "auto_fuel",
    "teleop_fuel",
    "endgame_fuel",
    "endgame_tower",
]


def get_tba_headers() -> Dict[str, str]:
    key = "uqTThWSrIgK7D7M3ct9fnwfIrj9m7ZzuCjwsgWsHzMtRl2xRNIm8pEQXVhfwOsBv"
    if not key:
        raise RuntimeError(
            "Set the TBA_AUTH_KEY environment variable to a valid The Blue Alliance authorization key."
        )
    return {"X-TBA-Auth-Key": key}


def flatten_dict(data: Dict[str, Any], parent_key: str = "", sep: str = "_") -> Dict[str, Any]:
    items: Dict[str, Any] = {}
    for key, value in (data or {}).items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else key
        if isinstance(value, dict):
            items.update(flatten_dict(value, new_key, sep=sep))
        elif isinstance(value, list):
            items[new_key] = "|".join(str(entry) for entry in value)
        else:
            items[new_key] = value
    return items


def team_key_to_number(team_key: Any) -> Any:
    if isinstance(team_key, str) and team_key.lower().startswith("frc"):
        suffix = team_key[3:]
        if suffix.isdigit():
            return int(suffix)
        return suffix
    return team_key


def request_tba(endpoint: str) -> Any:
    url = f"{TBA_BASE_URL}{endpoint}"
    response = requests.get(url, headers=get_tba_headers(), timeout=45)
    response.raise_for_status()
    return response.json()


def fetch_tba_matches(event_key: str) -> List[Dict[str, Any]]:
    try:
        return request_tba(f"/event/{event_key}/matches")
    except requests.RequestException as exc:
        raise RuntimeError("Failed to fetch matches from The Blue Alliance.") from exc


def fetch_statbotics_matches(event_key: str) -> List[Dict[str, Any]]:
    try:
        return sb_client.get_matches(event=event_key, limit=500, fields=["all"])
    except requests.RequestException as exc:
        raise RuntimeError("Could not reach the Statbotics API.") from exc
    except UserWarning as exc:
        raise RuntimeError(str(exc)) from exc
    except ValueError as exc:
        raise RuntimeError(str(exc)) from exc


def fetch_statbotics_team_matches(event_key: str) -> List[Dict[str, Any]]:
    matches: List[Dict[str, Any]] = []
    offset = 0
    while True:
        try:
            batch = sb_client.get_team_matches(
                event=event_key,
                limit=TEAM_MATCH_BATCH_SIZE,
                offset=offset,
                fields=["all"],
            )
        except requests.RequestException as exc:
            raise RuntimeError("Could not reach the Statbotics API for team_match data.") from exc
        except UserWarning as exc:
            raise RuntimeError(str(exc)) from exc
        except ValueError as exc:
            raise RuntimeError(str(exc)) from exc
        if not batch:
            break
        matches.extend(batch)
        if len(batch) < TEAM_MATCH_BATCH_SIZE:
            break
        offset += len(batch)
    return matches


def normalize_match_key(raw: Dict[str, Any]) -> Any:
    for key in ("match", "key", "match_key"):
        value = raw.get(key)
        if value:
            return value
    return None


def index_team_matches(team_matches: List[Dict[str, Any]]) -> Dict[str, Dict[Any, Dict[str, Any]]]:
    mapping: Dict[str, Dict[Any, Dict[str, Any]]] = {}
    for entry in team_matches:
        match_key = normalize_match_key(entry)
        if not match_key:
            continue
        team_num = team_key_to_number(entry.get("team"))
        if team_num is None:
            continue
        mapping.setdefault(match_key, {})[team_num] = entry
    return mapping


def build_base_row(match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    alliance_info = match.get("alliances", {}).get(alliance, {})
    team_keys = alliance_info.get("team_keys") or []
    row: Dict[str, Any] = {
        "match_key": match.get("key") or normalize_match_key(match),
        "event_key": match.get("event_key"),
        "set_number": match.get("set_number"),
        "match_number": match.get("match_number"),
        "alliance": alliance,
    }
    for idx in range(3):
        team_key = team_keys[idx] if idx < len(team_keys) else None
        row[f"team{idx + 1}"] = team_key_to_number(team_key)
    score = alliance_info.get("score")
    if score is not None:
        row[f"tba_{alliance}_score"] = score
    return row


def add_statbotics_metrics(row: Dict[str, Any], sb_match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    base = dict(row)
    base["sb_data_available"] = bool(sb_match)
    if not sb_match:
        return base
    flat = flatten_dict(sb_match)
    for key, value in flat.items():
        if key.startswith(alliance):
            base[f"sb_{key}"] = value
    alliance_data = sb_match.get(alliance)
    if isinstance(alliance_data, dict):
        flat_alliance = flatten_dict(alliance_data)
        for key, value in flat_alliance.items():
            base[f"sb_{alliance}_{key}"] = value
    predictions = sb_match.get("predictions") or {}
    alliance_predictions = predictions.get(alliance)
    if isinstance(alliance_predictions, dict):
        flat_preds = flatten_dict(alliance_predictions)
        for key, value in flat_preds.items():
            base[f"sb_{alliance}_prediction_{key}"] = value
    return base


def add_team_metrics(
    row: Dict[str, Any], match_key: str, team_match_map: Dict[str, Dict[Any, Dict[str, Any]]]
) -> Dict[str, Any]:
    base = dict(row)
    teams = team_match_map.get(match_key) or {}
    for idx in range(1, 4):
        team_value = row.get(f"team{idx}")
        if team_value is None:
            continue
        team_data = teams.get(team_value)
        if not isinstance(team_data, dict):
            continue
        flat = flatten_dict(team_data)
        for key, value in flat.items():
            base[f"sb_team{idx}_{key}"] = value
    return base


def add_tba_breakdown(row: Dict[str, Any], match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    base = dict(row)
    breakdown = (match.get("score_breakdown") or {}).get(alliance, {})
    flat_breakdown = flatten_dict(breakdown)
    for key, value in flat_breakdown.items():
        base[f"tba_{alliance}_{key}"] = value
    return base


def write_csv(rows: List[Dict[str, Any]], dest: Path) -> Path:
    if not rows:
        raise RuntimeError("No rows were collected for the requested event.")
    fieldnames = sorted({key for row in rows for key in row})
    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return dest


def export_event_data(event_key: str, *, output_dir: Path = Path(".")) -> Tuple[Path, Path]:
    tba_matches = fetch_tba_matches(event_key)
    if not tba_matches:
        raise RuntimeError(f"The Blue Alliance returned zero matches for {event_key}.")
    sb_matches = fetch_statbotics_matches(event_key)
    sb_team_matches = fetch_statbotics_team_matches(event_key)
    team_match_map = index_team_matches(sb_team_matches)
    sb_by_key = {
        normalize_match_key(entry): entry
        for entry in sb_matches
        if normalize_match_key(entry)
    }
    tba_matches.sort(
        key=lambda match: (
            match.get("set_number") or 0,
            match.get("match_number") or 0,
            match.get("key") or "",
        )
    )
    sb_rows: List[Dict[str, Any]] = []
    tba_rows: List[Dict[str, Any]] = []
    for match in tba_matches:
        match_key = match.get("key") or normalize_match_key(match)
        if not match_key:
            continue
        sb_match = sb_by_key.get(match_key)
        for alliance in ("red", "blue"):
            base = build_base_row(match, alliance)
            sb_row = add_statbotics_metrics(base, sb_match, alliance)
            sb_rows.append(add_team_metrics(sb_row, match_key, team_match_map))
            tba_rows.append(add_tba_breakdown(base, match, alliance))
    sb_path = write_csv(sb_rows, output_dir / "sb.csv")
    tba_path = write_csv(tba_rows, output_dir / "tba.csv")
    return sb_path, tba_path
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

import requests
from statbotics import Statbotics

TBA_BASE_URL = "https://www.thebluealliance.com/api/v3"
sb_client = Statbotics()


def get_tba_headers() -> Dict[str, str]:
    key = "uqTThWSrIgK7D7M3ct9fnwfIrj9m7ZzuCjwsgWsHzMtRl2xRNIm8pEQXVhfwOsBv"
    if not key:
        raise RuntimeError(
            "Set the TBA_AUTH_KEY environment variable to a valid The Blue Alliance authorization key."
        )
    return {"X-TBA-Auth-Key": key}


def flatten_dict(data: Dict[str, Any], parent_key: str = "", sep: str = "_") -> Dict[str, Any]:
    items: Dict[str, Any] = {}
    for key, value in (data or {}).items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else key
        if isinstance(value, dict):
            items.update(flatten_dict(value, new_key, sep=sep))
        elif isinstance(value, list):
            items[new_key] = "|".join(str(entry) for entry in value)
        else:
            items[new_key] = value
    return items


def team_key_to_number(team_key: Any) -> Any:
    if isinstance(team_key, str) and team_key.lower().startswith("frc"):
        suffix = team_key[3:]
        if suffix.isdigit():
            return int(suffix)
        return suffix
    return team_key


def request_tba(endpoint: str) -> Any:
    url = f"{TBA_BASE_URL}{endpoint}"
    response = requests.get(url, headers=get_tba_headers(), timeout=45)
    response.raise_for_status()
    return response.json()


def fetch_tba_matches(event_key: str) -> List[Dict[str, Any]]:
    try:
        return request_tba(f"/event/{event_key}/matches")
    except requests.RequestException as exc:
        raise RuntimeError("Failed to fetch matches from The Blue Alliance.") from exc


def fetch_statbotics_matches(event_key: str) -> List[Dict[str, Any]]:
    try:
        return sb_client.get_matches(event=event_key, limit=500, fields=["all"])
    except requests.RequestException as exc:
        raise RuntimeError("Could not reach the Statbotics API.") from exc
    except UserWarning as exc:
        raise RuntimeError(str(exc)) from exc
    except ValueError as exc:
        raise RuntimeError(str(exc)) from exc

def normalize_match_key(raw: Dict[str, Any]) -> Any:
    for key in ("match", "key", "match_key"):
        value = raw.get(key)
        if value:
            return value
    return None


def build_base_row(match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    alliance_info = match.get("alliances", {}).get(alliance, {})
    team_keys = alliance_info.get("team_keys") or []
    row: Dict[str, Any] = {
        "match_key": match.get("key") or normalize_match_key(match),
        "event_key": match.get("event_key"),
        "set_number": match.get("set_number"),
        "match_number": match.get("match_number"),
        "alliance": alliance,
    }
    for idx in range(3):
        team_key = team_keys[idx] if idx < len(team_keys) else None
        row[f"team{idx + 1}"] = team_key_to_number(team_key)
    score = alliance_info.get("score")
    if score is not None:
        row[f"tba_{alliance}_score"] = score
    return row


def add_statbotics_metrics(row: Dict[str, Any], sb_match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    base = dict(row)
    base["sb_data_available"] = bool(sb_match)
    if not sb_match:
        return base
    flat = flatten_dict(sb_match)
    for key, value in flat.items():
        if key.startswith(alliance):
            base[f"sb_{key}"] = value
    alliance_data = sb_match.get(alliance)
    if isinstance(alliance_data, dict):
        flat_alliance = flatten_dict(alliance_data)
        for key, value in flat_alliance.items():
            base[f"sb_{alliance}_{key}"] = value
    predictions = sb_match.get("predictions") or {}
    alliance_predictions = predictions.get(alliance)
    if isinstance(alliance_predictions, dict):
        flat_preds = flatten_dict(alliance_predictions)
        for key, value in flat_preds.items():
            base[f"sb_{alliance}_prediction_{key}"] = value
    return base


def add_tba_breakdown(row: Dict[str, Any], match: Dict[str, Any], alliance: str) -> Dict[str, Any]:
    base = dict(row)
    breakdown = (match.get("score_breakdown") or {}).get(alliance, {})
    flat_breakdown = flatten_dict(breakdown)
    for key, value in flat_breakdown.items():
        base[f"tba_{alliance}_{key}"] = value
    return base


def write_csv(rows: List[Dict[str, Any]], dest: Path) -> Path:
    if not rows:
        raise RuntimeError("No rows were collected for the requested event.")
    fieldnames = sorted({key for row in rows for key in row})
    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return dest


def export_event_data(event_key: str, *, output_dir: Path = Path(".")) -> Tuple[Path, Path]:
    tba_matches = fetch_tba_matches(event_key)
    if not tba_matches:
        raise RuntimeError(f"The Blue Alliance returned zero matches for {event_key}.")
    sb_matches = fetch_statbotics_matches(event_key)
    sb_by_key = {
        normalize_match_key(entry): entry
        for entry in sb_matches
        if normalize_match_key(entry)
    }
    tba_matches.sort(
        key=lambda match: (
            match.get("set_number") or 0,
            match.get("match_number") or 0,
            match.get("key") or "",
        )
    )
    sb_rows: List[Dict[str, Any]] = []
    tba_rows: List[Dict[str, Any]] = []
    for match in tba_matches:
        match_key = match.get("key") or normalize_match_key(match)
        if not match_key:
            continue
        sb_match = sb_by_key.get(match_key)
        for alliance in ("red", "blue"):
            base = build_base_row(match, alliance)
            sb_rows.append(add_statbotics_metrics(base, sb_match, alliance))
            tba_rows.append(add_tba_breakdown(base, match, alliance))
    sb_path = write_csv(sb_rows, output_dir / "sb.csv")
    tba_path = write_csv(tba_rows, output_dir / "tba.csv")
    return sb_path, tba_path


In [ ]:
event_key = input("2026orsal" ).strip()
if not event_key:
    raise ValueError("Please provide an event key before running the exporter.")
sb_path, tba_path = export_event_data(event_key)
print(f"Generated {sb_path.name} and {tba_path.name} in {sb_path.parent.resolve()}") #bad no not use


Generated sb.csv and tba.csv in C:\Users\Aarush\Documents\FRC\2026Scripts\betterSB


: 

- sb.csv contains one row per match+alliance pairing with the alliance teams and every Statbotics metric the API exposes (EPA, auto/teleop/endgame components, predictions, etc.).
- 	ba.csv contains the same match/alliance metadata plus each alliance's The Blue Alliance score breakdown (fuel/hub counts, fouls, etc.).
- Re-run the runner cell with a different key to refresh either file for another event.
